# Chenopodium plant segmentation — geometry-only pipeline

*Converted from `script/script_test_sansviterbi2.py`.*

*Archived source: `script/script_test_sansviterbi2.py` @ commit `428f576` — restore with `git show 428f576:script/script_test_sansviterbi2.py`.*


This notebook reproduces the full automatic plant-segmentation pipeline on a single point cloud
(`data/chenos/cheno_3.ply`). It builds a connected Riemannian graph from the cloud, computes the
Fiedler vector / gradient, clusters the points, builds a quotient graph, and then runs a sequence of
**geometry-only** semantic refinements (leaves / stem / apex / petioles / branches) followed by a
topology-control pass.

The pipeline is intentionally **data-agnostic**: the root (lowest) point is detected automatically
from the minimum `z` coordinate, so the same cells run on any `cheno_*.ply` cloud.

> Run the cells top to bottom. Slow steps (kNN graph construction, eigen-decomposition, elbow
> resegmentation, energy optimisation) are marked and may take several seconds to a couple of minutes.


## 1. Imports

In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import pandas as pd

import scipy.sparse as spsp
import scipy as sp
import sklearn as sk
import spectral_clustering.similarity_graph as sgk
import open3d as o3d

import spectral_clustering.point_cloud_graph as kpcg

import time
import copy
import os

from collections import Counter

from spectral_clustering.point_cloud_graph import *
from spectral_clustering.quotient_graph import *
from spectral_clustering.topological_energy import *
from spectral_clustering.segmentation_algorithms import *
from spectral_clustering.split_and_merge import *
from spectral_clustering.display_and_export import *
from spectral_clustering.quotientgraph_semantics import *
from spectral_clustering.dijkstra_segmentation import *
from spectral_clustering.quotientgraph_operations import *
from spectral_clustering.evaluation import *

from importlib import reload


In [ ]:
# All outputs (figures, .txt exports) are written to the local `outputs/`
# sub-folder, which is git-ignored (see `.gitignore`). Library export
# functions receive `OUTPUT_DIR` explicitly instead of relying on a global
# `os.chdir`.
from pathlib import Path
OUTPUT_DIR = Path.cwd() / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

## 2. Load point cloud & build the connected Riemannian graph

The original script took a filename as a command-line argument; here we hard-code the local cloud
`../data/chenos/cheno_3.ply` (the notebook runs with cwd = `notebooks/`). `r = 18` is the kNN radius
used to build the sparse connected graph.

This is the **slowest** step: ~290k edges on 32k points.


In [ ]:
# @time (10-30s): kNN graph construction
begin = time.time()
model_filename = '../data/chenos/cheno_3.ply'
filenamemod = os.path.splitext(os.path.basename(model_filename))[0]
print('cloud:', model_filename)

pcd = o3d.io.read_point_cloud(model_filename, format='ply')
r = 18
SimG, pcdfinal = sgk.create_connected_riemannian_graph(point_cloud=pcd, method='knn', nearest_neighbors=r)
G = PointCloudGraph(SimG)
G.pcd = pcdfinal
print(f'points: {len(G.nodes)}, edges: {G.number_of_edges()}')


## 3. Detect the lowest point (root)

The root of the plant is the point with the minimal `z` coordinate (lowest in the cloud). This is
computed automatically here; the original script allowed a manual override.


In [ ]:
def detect_lower_point(G):
    a = np.amin(G.nodes_coords[:, 2], axis=0)
    result = np.where(G.nodes_coords[:, 2] == a)
    return result[0][0]

root_point_riemanian = detect_lower_point(G)
print('root point index:', root_point_riemanian)


## 4. Spectral decomposition of the graph Laplacian

Compute the Laplacian eigenvectors (Fiedler vector = the 2nd smallest eigenvalue's eigenvector),
then the gradient of the Fiedler vector, then cluster the points into 4 classes by k-means on the
gradient norm. This coarse clustering is the seed for the quotient graph.

`@time`: eigen-decomposition ~10s, gradient ~4s, k-means ~2s.


In [ ]:
# @time (~15s)
G.compute_graph_eigenvectors()

export_fiedler_vector_on_pointcloud(G, filename="pcd_vp2.txt", directory=OUTPUT_DIR)
G.compute_gradient_of_fiedler_vector(method='by_fiedler_weight')
clusters_centers = G.clustering_by_kmeans_using_gradient_norm(export_in_labeled_point_cloud=True, number_of_clusters=4, directory=OUTPUT_DIR)
print('cluster centers:', clusters_centers)


## 5. Build the quotient graph

Each connected component of a k-means cluster becomes a node of the quotient graph (`QG`). Point-cloud
nodes keep a `quotient_graph_node` attribute pointing at their QG node.


In [ ]:
QG = QuotientGraph()
QG.build_from_pointcloudgraph(G, G.kmeans_labels_gradient)
print('quotient graph nodes:', len(QG.nodes))

export_some_graph_attributes_on_point_cloud(QG.point_cloud_graph,
                                            graph_attribute="quotient_graph_node",
                                            filename="pcd_attribute.txt", directory=OUTPUT_DIR)


## 6. Geometry-only semantics — norm 1

Leaves are taken as the QG class with the **minimum centroid** (lowest average height); everything else
is considered "linear" (stem / petioles / branches). This coarse labelling is stored in
`viterbi_class` (1 = leaf, 0 = linear).


In [ ]:
label_leaves = select_minimum_centroid_class(clusters_centers)
list_leaves1 = select_all_quotientgraph_nodes_from_pointcloudgraph_cluster(G, QG, label_leaves)

list_of_linear = []
for n in QG.nodes:
    QG.nodes[n]["viterbi_class"] = np.nan
for n in QG.nodes:
    if n in list_leaves1:
        QG.nodes[n]["viterbi_class"] = 1
    if n not in list_leaves1:
        list_of_linear.append(n)
        QG.nodes[n]["viterbi_class"] = 0
if G.nodes[root_point_riemanian]["quotient_graph_node"] in list_leaves1:
    list_leaves1.remove(G.nodes[root_point_riemanian]["quotient_graph_node"])
list_of_linear.append(G.nodes[root_point_riemanian]["quotient_graph_node"])
G.nodes[root_point_riemanian]["viterbi_class"] = 0

export_quotient_graph_attribute_on_point_cloud(QG, attribute='viterbi_class', name='semantic_from_norm1', directory=OUTPUT_DIR)


## 7. Resegment the linear nodes (norm 2)

The "linear" QG nodes are re-segmented with the elbow method using the **direction** of the Fiedler
gradient (`direction_gradient`, attribute 3), then the quotient graph is rebuilt. Leaves are re-selected
and `viterbi_class` is re-assigned.

`@time`: elbow resegmentation ~10s.


In [ ]:
# @time (~10s)
list_leaves_point = []
for qnode in list_leaves1:
    list_of_nodes_each = [x for x, y in G.nodes(data=True) if y['quotient_graph_node'] == qnode]
    list_leaves_point += list_of_nodes_each

list_of_linear = []
for n in QG.nodes:
    if n not in list_leaves1:
        list_of_linear.append(n)

resegment_nodes_with_elbow_method(QG, QG_nodes_to_rework=list_of_linear, number_of_cluster_tested=20,
                                  attribute='direction_gradient', number_attribute=3, standardization=False)
QG.rebuild(QG.point_cloud_graph)

export_some_graph_attributes_on_point_cloud(QG.point_cloud_graph,
                                            graph_attribute="quotient_graph_node",
                                            filename="reseg_linear.txt", directory=OUTPUT_DIR)

list_leaves2 = select_all_quotientgraph_nodes_from_pointcloudgraph_cluster(G, QG, label_leaves)
list_of_linear = []
for n in QG.nodes:
    QG.nodes[n]["viterbi_class"] = np.nan
for n in QG.nodes:
    if n in list_leaves2:
        QG.nodes[n]["viterbi_class"] = 1
    if n not in list_leaves2:
        list_of_linear.append(n)
        QG.nodes[n]["viterbi_class"] = 0
if G.nodes[root_point_riemanian]["quotient_graph_node"] in list_leaves2:
    list_leaves2.remove(G.nodes[root_point_riemanian]["quotient_graph_node"])
list_of_linear.append(G.nodes[root_point_riemanian]["quotient_graph_node"])
G.nodes[root_point_riemanian]["viterbi_class"] = 0

export_quotient_graph_attribute_on_point_cloud(QG, attribute='viterbi_class', name='semantic_from_norm2', directory=OUTPUT_DIR)


## 8. Direction information & energy optimisation (norm 3)

Compute per-node direction info, then run the `opti_energy_dot_product` energy optimisation, which
progressively splits QG nodes to minimise a dot-product energy. The quotient graph is rebuilt and
`viterbi_class` is re-assigned once more.

> **Gotcha:** `QG.rebuild(..., clear=True)` wipes the `viterbi_class` attribute, so it **must** be
> re-assigned *after* the rebuild before the semantic stage below.

`@time`: energy optimisation ~15s.


In [ ]:
# @time (~15s)
QG.compute_direction_info(list_leaves=list_leaves2)
opti_energy_dot_product(quotientgraph=QG, subgraph_riemannian=G, angle_to_stop=30, export_iter=True,
                        list_leaves=list_leaves2, directory=OUTPUT_DIR)
QG.rebuild(QG.point_cloud_graph)
G = QG.point_cloud_graph

list_leaves3 = select_all_quotientgraph_nodes_from_pointcloudgraph_cluster(G, QG, label_leaves)
list_of_linear = []
for n in QG.nodes:
    QG.nodes[n]["viterbi_class"] = np.nan
for n in QG.nodes:
    if n in list_leaves3:
        QG.nodes[n]["viterbi_class"] = 1
    if n not in list_leaves3:
        list_of_linear.append(n)
        QG.nodes[n]["viterbi_class"] = 0
if G.nodes[root_point_riemanian]["quotient_graph_node"] in list_leaves3:
    list_leaves3.remove(G.nodes[root_point_riemanian]["quotient_graph_node"])
list_of_linear.append(G.nodes[root_point_riemanian]["quotient_graph_node"])
G.nodes[root_point_riemanian]["viterbi_class"] = 0

export_quotient_graph_attribute_on_point_cloud(QG, attribute='viterbi_class', name='semantic_from_norm3', directory=OUTPUT_DIR)


## 9. End of geometry-only — botanical semantics

From here on we label the QG with **botanical** classes. First store node centroids, then:

* **Apex** vs **limb** (`differenciate_apex_limb`, new class 4),
* **Stem** detection via leaf quotient (`stem_detection_with_quotite_leaves`, new class 3),
* merge stem nodes,
* compute **leaf quotients**,
* weight edges semantically and run **shortest paths** to detect **branches** (class 6) and **petioles** (class 5),
* merge remaining clusters and remove non-useful edges.


In [ ]:
# Centroids of QG nodes (stored in 'centroide_coordinates')
QG.ponderate_with_coordinates_of_points()

list_leaves3 = [x for x, y in QG.nodes(data=True) if y['viterbi_class'] == 1]
list_of_linear = [x for x, y in QG.nodes(data=True) if y['viterbi_class'] == 0]


In [ ]:
# Apex / limb discrimination
differenciate_apex_limb(QG, attribute_class='viterbi_class', number_leaves_limb=1, new_apex_class=4)
export_quotient_graph_attribute_on_point_cloud(QG, attribute='viterbi_class', name='semantic_apex_limb', directory=OUTPUT_DIR)
display_and_export_quotient_graph_matplotlib(qg=QG, node_sizes=20, name="apex_classlqg2",
                                             data_on_nodes='viterbi_class', data=True, attributekmeans4clusters=False, directory=OUTPUT_DIR)
export_some_graph_attributes_on_point_cloud(QG.point_cloud_graph,
                                            graph_attribute="quotient_graph_node",
                                            filename="pcd_seg.txt", directory=OUTPUT_DIR)


In [ ]:
# Stem detection + merge stem nodes
list_apex = [x for x, y in QG.nodes(data=True) if y['viterbi_class'] == 4]
stem_detection_with_quotite_leaves(QG, list_leaves3, list_apex, list_of_linear, root_point_riemanian, new_class_stem=3, directory=OUTPUT_DIR)

QG = merge_one_class_QG_nodes(QG, attribute='viterbi_class', viterbiclass=[3])
QG.point_cloud_graph = G
display_and_export_quotient_graph_matplotlib(qg=QG, node_sizes=20, name="QG_merge_tige",
                                             data_on_nodes='viterbi_class', data=True, attributekmeans4clusters=False, directory=OUTPUT_DIR)


In [ ]:
# Leaf quotients
list_leaves = [x for x, y in QG.nodes(data=True) if y['viterbi_class'] == 1]
list_notleaves = [x for x, y in QG.nodes(data=True) if y['viterbi_class'] != 1]
calculate_leaf_quotients(QG, list_leaves, list_notleaves, root_point_riemanian)
for n in list_leaves:
    QG.nodes[n]["leaf_quotient_n"] = -1


In [ ]:
# Semantic weights + shortest paths -> branches
weight_with_semantic(QG=QG, class_attribute='viterbi_class', class_limb=1, class_mainstem=3, class_linear=0,
                     class_apex=4,
                     weight_limb_limb=10000, weight_apex_apex=10000, weight_apex_mainstem=10000,
                     weight_limb_apex=10000, weight_limb_mainstem=10000, weight_linear_mainstem=0,
                     weight_linear_linear=0, weight_linear_limb=0, weight_linear_apex=0)

shortest_paths_from_apex_det_branch(QG, root_point_riemanian, class_attribute='viterbi_class',
                                    weight='weight_sem_paths', class_apex=4, class_branch=0, class_mainstem=3,
                                    new_class_branch=6)

display_and_export_quotient_graph_matplotlib(qg=QG, node_sizes=20, name="apex_class_pet_branches",
                                             data_on_nodes='viterbi_class', data=True, attributekmeans4clusters=False, directory=OUTPUT_DIR)
export_quotient_graph_attribute_on_point_cloud(QG, attribute='viterbi_class', name='semantic_apex_limb_petiole_branches', directory=OUTPUT_DIR)


In [ ]:
# Semantic weights + shortest paths -> petioles
maj_weight_semantics(QG, class_attribute='viterbi_class', class_limb=1, class_mainstem=3, class_petiol=5,
                     class_apex=4, class_branch=6,
                     weight_petiol_petiol=0, weight_petiol_apex=10000, weight_branch_limb=10000)

shortest_paths_from_limbs_det_petiol(QG, root_point_riemanian, class_attribute='viterbi_class',
                                     weight='weight_sem_paths', class_limb=1, class_linear=0, class_mainstem=3,
                                     new_class_petiol=5)

display_and_export_quotient_graph_matplotlib(qg=QG, node_sizes=20, name="apex_class_pet",
                                             data_on_nodes='viterbi_class', data=True, attributekmeans4clusters=False, directory=OUTPUT_DIR)
export_quotient_graph_attribute_on_point_cloud(QG, attribute='viterbi_class', name='semantic_apex_limb_petiole', directory=OUTPUT_DIR)


In [ ]:
# Merge remaining clusters (class 0) and remove non-useful edges
QG = merge_remaining_clusters(quotientgraph=QG, remaining_clusters_class=0, class_attribute='viterbi_class')
display_and_export_quotient_graph_matplotlib(qg=QG, node_sizes=20, name="apex_class_pet_branches_delete0",
                                             data_on_nodes='viterbi_class', data=True, attributekmeans4clusters=False, directory=OUTPUT_DIR)
export_quotient_graph_attribute_on_point_cloud(QG, attribute='viterbi_class', name='semantic_apex_limb_petiole_branches_delete0', directory=OUTPUT_DIR)

remove_edges_useful_paths(QG)
display_and_export_quotient_graph_matplotlib(qg=QG, node_sizes=20, name="apex_class_pet_branches_remove",
                                             data_on_nodes='viterbi_class', data=True, attributekmeans4clusters=False, directory=OUTPUT_DIR)
export_quotient_graph_attribute_on_point_cloud(QG, attribute='viterbi_class', name='semantic_apex_limb_petiole_branches_remove', directory=OUTPUT_DIR)

display_and_export_quotient_graph_matplotlib(qg=QG, node_sizes=20, name=filenamemod + "_semantic_1",
                                             data_on_nodes='viterbi_class', data=True, attributekmeans4clusters=False, directory=OUTPUT_DIR)
export_quotient_graph_attribute_on_point_cloud(QG, attribute='viterbi_class', name=filenamemod + '_semantic_1', directory=OUTPUT_DIR)
export_some_graph_attributes_on_point_cloud(QG.point_cloud_graph,
                                            graph_attribute="quotient_graph_node",
                                            filename=filenamemod + "_instance_1.txt", directory=OUTPUT_DIR)


## 10. Topology control

`topology_control` flags structurally inconsistent nodes by tagging them with error codes
(`error_norm = 10`, `error_dir = 11`). We then treat each error with `treat_topology_error2`:
* error 10 → re-segment the node with the **norm** gradient and re-classify its points as limb / linear,
* error 11 → re-segment with the **direction** gradient and force to linear.

Finally the point-cloud-level `viterbi_class` is refreshed via
`transfer_quotientgraph_infos_on_riemanian_graph`.


In [ ]:
topology_control(QG, attribute_class_control='viterbi_class',
                 class_stem=3, class_petiols=5, class_limb=1,
                 class_apex=4, error_norm=10, error_dir=11)

display_and_export_quotient_graph_matplotlib(qg=QG, node_sizes=20, name="topo_control",
                                             data_on_nodes='viterbi_class', data=True, attributekmeans4clusters=False, directory=OUTPUT_DIR)
export_quotient_graph_attribute_on_point_cloud(QG, attribute='viterbi_class', name='topo_control', directory=OUTPUT_DIR)

transfer_quotientgraph_infos_on_riemanian_graph(QG, info='viterbi_class')


In [ ]:
def treat_topology_error2(QG, attribute_class_control='viterbi_class', number_attribute=1,
                          class_limb=1, class_linear=0, error=10, way_to_treat='norm_gradient',
                          number_of_cluster_tested=20):
    list_nodes_QG_to_work = [x for x, y in QG.nodes(data=True) if y[attribute_class_control] == error]
    G = QG.point_cloud_graph
    for node_QG in list_nodes_QG_to_work:
        SG_node = create_subgraphs_to_work(QG, list_quotient_node_to_work=[node_QG])

        clusters_centers = resegment_nodes_with_elbow_method(
            QG, QG_nodes_to_rework=[node_QG], number_of_cluster_tested=number_of_cluster_tested,
            attribute=way_to_treat, number_attribute=number_attribute, standardization=False, numer=1)

        if way_to_treat == 'norm_gradient':
            label_leaves = select_minimum_centroid_class(clusters_centers)
            list_true_labels = list(set(nx.get_node_attributes(SG_node, 'quotient_graph_node').values()))
            list_true_labels.sort()
            points_leaves = [x for x, y in G.nodes(data=True) if y['quotient_graph_node'] == list_true_labels[label_leaves]]
            for n in SG_node:
                if n in points_leaves:
                    G.nodes[n][attribute_class_control] = class_limb
                else:
                    G.nodes[n][attribute_class_control] = class_linear

        if way_to_treat == 'direction_gradient':
            for n in SG_node:
                G.nodes[n][attribute_class_control] = class_linear


In [ ]:
# Treat norm errors (error = 10)
treat_topology_error2(QG=QG, attribute_class_control='viterbi_class', number_attribute=1,
                      class_limb=1, class_linear=0, error=10, way_to_treat='norm_gradient',
                      number_of_cluster_tested=20)
export_some_graph_attributes_on_point_cloud(QG.point_cloud_graph,
                                            graph_attribute="quotient_graph_node",
                                            filename="pcd_seg_norm.txt", directory=OUTPUT_DIR)

# Treat direction errors (error = 11)
treat_topology_error2(QG=QG, attribute_class_control='viterbi_class', number_attribute=3,
                      class_limb=1, class_linear=0, error=11, way_to_treat='direction_gradient',
                      number_of_cluster_tested=20)
export_some_graph_attributes_on_point_cloud(QG.point_cloud_graph,
                                            graph_attribute="quotient_graph_node",
                                            filename="pcd_seg_dir.txt", directory=OUTPUT_DIR)

export_some_graph_attributes_on_point_cloud(QG.point_cloud_graph,
                                            graph_attribute="quotient_graph_node",
                                            filename="pcd_reseg.txt", directory=OUTPUT_DIR)
export_some_graph_attributes_on_point_cloud(QG.point_cloud_graph,
                                            graph_attribute="viterbi_class",
                                            filename="pcd_reseg_semantic.txt", directory=OUTPUT_DIR)


## 11. Final botanical tree

Rebuild the QG from the corrected labels and produce the final botanical hierarchy with
`obtain_tree_with_botanical` (apex/limb discrimination, stem detection, branch and petiole shortest
paths, merge remaining clusters, remove non-useful edges).


In [ ]:
QG.rebuild(G)
label_leaves = 1.0
list_leaves3 = select_all_quotientgraph_nodes_from_pointcloudgraph_cluster(G, QG, label_leaves, attribute='viterbi_class')
list_apex3 = select_all_quotientgraph_nodes_from_pointcloudgraph_cluster(G, QG, 4.0, attribute='viterbi_class')
for n in QG.nodes:
    if n in list_leaves3:
        QG.nodes[n]["viterbi_class"] = 1
    elif n in list_apex3:
        QG.nodes[n]["viterbi_class"] = 1
    else:
        list_of_linear.append(n)
        QG.nodes[n]["viterbi_class"] = 0


In [ ]:
def obtain_tree_with_botanical(QG):
    G = QG.point_cloud_graph
    differenciate_apex_limb(QG, attribute_class='viterbi_class', number_leaves_limb=1, new_apex_class=4)
    list_apex = [x for x, y in QG.nodes(data=True) if y['viterbi_class'] == 4]
    list_of_linear = [x for x, y in QG.nodes(data=True) if y['viterbi_class'] == 0]
    stem_detection_with_quotite_leaves(QG, list_leaves3, list_apex, list_of_linear, root_point_riemanian, new_class_stem=3, directory=OUTPUT_DIR)
    QG = merge_one_class_QG_nodes(QG, attribute='viterbi_class', viterbiclass=[3])
    QG.point_cloud_graph = G
    weight_with_semantic(QG=QG, class_attribute='viterbi_class', class_limb=1, class_mainstem=3, class_linear=0,
                         class_apex=4,
                         weight_limb_limb=10000, weight_apex_apex=10000, weight_apex_mainstem=10000,
                         weight_limb_apex=10000, weight_limb_mainstem=10000, weight_linear_mainstem=0,
                         weight_linear_linear=0, weight_linear_limb=0, weight_linear_apex=0)
    shortest_paths_from_apex_det_branch(QG, root_point_riemanian, class_attribute='viterbi_class',
                                        weight='weight_sem_paths', class_apex=4, class_branch=0, class_mainstem=3,
                                        new_class_branch=6)
    maj_weight_semantics(QG, class_attribute='viterbi_class', class_limb=1, class_mainstem=3, class_petiol=5,
                         class_apex=4, class_branch=6,
                         weight_petiol_petiol=0, weight_petiol_apex=10000, weight_branch_limb=10000)
    shortest_paths_from_limbs_det_petiol(QG, root_point_riemanian, class_attribute='viterbi_class',
                                         weight='weight_sem_paths', class_limb=1, class_linear=0, class_mainstem=3,
                                         new_class_petiol=5)
    QG = merge_remaining_clusters(quotientgraph=QG, remaining_clusters_class=0, class_attribute='viterbi_class')
    remove_edges_useful_paths(QG)
    return QG

QG = obtain_tree_with_botanical(QG)
print('final botanical tree, QG nodes:', len(QG.nodes))


In [ ]:
display_and_export_quotient_graph_matplotlib(qg=QG, node_sizes=20, name=filenamemod + "_semantic_final",
                                             data_on_nodes='viterbi_class', data=True, attributekmeans4clusters=False, directory=OUTPUT_DIR)
export_quotient_graph_attribute_on_point_cloud(QG, attribute='viterbi_class', name=filenamemod + '_semantic_final', directory=OUTPUT_DIR)
export_some_graph_attributes_on_point_cloud(QG.point_cloud_graph,
                                            graph_attribute="quotient_graph_node",
                                            filename=filenamemod + "_instance_final.txt", directory=OUTPUT_DIR)

time2 = time.time()
print('total elapsed: %.1fs' % (time2 - begin))
